# Gold Aggregation — Ejecución Manual
Notebook de respaldo para la capa Gold en caso de fallo en Airflow.

Lee `posts_hist` y `users_hist` desde Silver (Iceberg), cruza ambas tablas
y genera la tabla analítica `nessie.gold.cant_post_x_user_hist` con merge (sin override).

## 1. Configuración e imports

In [ ]:
import os
import pyspark
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, current_date, count, avg, sum as spark_sum,
    when, lit, round as spark_round, coalesce
)

# Requerido por el SDK de AWS aunque MinIO no use region real
os.environ["AWS_REGION"]            = "us-east-1"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"]     = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"

In [ ]:
# Parametros de conexion
CATALOG_URI    = "http://nessie:19120/api/v1"
WAREHOUSE      = "s3a://proyecto2/gold"
S3_ENDPOINT    = "http://minio:9000"
AWS_ACCESS_KEY = "admin"
AWS_SECRET_KEY = "password"

# Tablas fuente (Silver)
SILVER_NAMESPACE   = "nessie.silver"
USERS_TABLE        = f"{SILVER_NAMESPACE}.users_hist"
POSTS_TABLE        = f"{SILVER_NAMESPACE}.posts_hist"

# Tabla destino (Gold) - una sola tabla de metricas
GOLD_NAMESPACE     = "nessie.gold"
GOLD_METRICS_TABLE = f"{GOLD_NAMESPACE}.cant_post_x_user_hist"

## 2. Inicializar SparkSession

In [ ]:
conf = (
    pyspark.SparkConf()
    .setAppName("gold_agg_notebook")
    .set("spark.driver.memory", "8g")
    .set("spark.executor.memory", "16g")
    .set("spark.sql.shuffle.partitions", "200")
    .set("spark.driver.maxResultSize", "4g")
    .set("spark.network.timeout", "800s")
    .set("spark.executor.heartbeatInterval", "60s")
    .set("spark.sql.broadcastTimeout", "1200")
    .set("spark.sql.autoBroadcastJoinThreshold", "104857600")
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .set(
        "spark.jars.packages",
        ",".join([
            "org.postgresql:postgresql:42.7.3",
            "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.0",
            "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.77.1",
            "software.amazon.awssdk:bundle:2.24.8",
            "software.amazon.awssdk:url-connection-client:2.24.8",
            "org.apache.hadoop:hadoop-aws:3.2.0",
        ]),
    )
    .set(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
        "org.projectnessie.spark.extensions.NessieSparkSessionExtensions",
    )
    # Catalogo Nessie
    .set("spark.sql.catalog.nessie",                        "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri",                    CATALOG_URI)
    .set("spark.sql.catalog.nessie.ref",                    "main")
    .set("spark.sql.catalog.nessie.authentication.type",    "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl",           "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse",              WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl",                "org.apache.iceberg.aws.s3.S3FileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint",            S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.path-style-access",  "true")
    .set("spark.sql.catalog.nessie.s3.access-key-id",      AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-access-key",  AWS_SECRET_KEY)
    .set("spark.sql.catalog.nessie.s3.region",             "us-east-1")
    # Hadoop S3A
    .set("spark.hadoop.fs.s3a.endpoint",                    S3_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key",                  AWS_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key",                  AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access",           "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider",   "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl",                       "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.connection.ssl.enabled",     "false")
    .set("spark.hadoop.fs.s3a.endpoint.region",            "us-east-1")
    .set("spark.driver.extraJavaOptions",
         "-Daws.region=us-east-1 -Daws.accessKeyId=admin -Daws.secretAccessKey=password")
    .set("spark.executor.extraJavaOptions",
         "-Daws.region=us-east-1 -Daws.accessKeyId=admin -Daws.secretAccessKey=password")
)

spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("SparkSession iniciada correctamente.")
spark

## 3. Funciones auxiliares

In [ ]:
def create_namespace_if_needed():
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {GOLD_NAMESPACE}")
    print(f"Namespace {GOLD_NAMESPACE} listo.")


def spark_table_exists(table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        return False


def merge_into_iceberg(df_new: DataFrame, table_name: str, key_col: str = "user_id") -> None:
    """
    MERGE manual con PySpark puro — mismo patron que Silver.
    Si la tabla no existe la crea; si existe, actualiza registros
    existentes e inserta los nuevos sin hacer override completo.
    """
    if not spark_table_exists(table_name):
        print(f"Tabla {table_name} no existe. Creando con carga inicial...")
        (
            df_new.writeTo(table_name)
            .using("iceberg")
            .tableProperty("format-version", "2")
            .create()
        )
        print(f"Tabla {table_name} creada correctamente.")
        return

    print(f"Leyendo tabla existente {table_name}...")
    df_existing = spark.read.format("iceberg").load(table_name)

    # Registros que NO vienen en el nuevo batch se conservan tal cual
    df_unchanged = df_existing.join(
        df_new.select(key_col),
        on=key_col,
        how="left_anti"
    )

    # Union: sin cambios + nuevo batch (cubre updates e inserts)
    df_merged = df_unchanged.unionByName(df_new, allowMissingColumns=True)

    print(f"Sobreescribiendo {table_name} con datos mergeados...")
    (
        df_merged.writeTo(table_name)
        .using("iceberg")
        .overwritePartitions()
    )
    print(f"MERGE completado en {table_name}.")

## 4. Crear namespace Gold

In [ ]:
create_namespace_if_needed()

## 5. Leer tablas Silver

In [ ]:
print("Leyendo tablas Silver...")

# Schema users_hist:
#   id, reputation, creation_date, display_name, last_access_date,
#   about_me, views, up_votes, down_votes, website_url, location,
#   account_id, fecha_cargue
df_users = spark.read.format("iceberg").load(USERS_TABLE)

# Schema posts_hist:
#   id, post_type_id, accepted_answer_id, creation_date, score,
#   view_count, body, owner_user_id, last_edit_date, last_activity_date,
#   title, tags, answer_count, comment_count, favorite_count,
#   content_license, parent_id, community_owned_date, closed_date, fecha_cargue
df_posts = spark.read.format("iceberg").load(POSTS_TABLE)

print(f"  users_hist : {df_users.count()} registros")
print(f"  posts_hist : {df_posts.count()} registros")

## 6. Agregar metricas de posts por usuario

Paso 1: se agrupan todos los posts por `owner_user_id` calculando
conteos, sumas y promedios sobre las columnas relevantes.

In [ ]:
print("Agregando metricas de posts por usuario...")

df_posts_agg = (
    df_posts
    .filter(col("owner_user_id").isNotNull())
    .groupBy(col("owner_user_id").alias("user_id"))
    .agg(
        # Volumetria general
        count("id").alias("total_posts"),
        count(when(col("post_type_id") == 1, True)).alias("total_preguntas"),
        count(when(col("post_type_id") == 2, True)).alias("total_respuestas"),

        # Metricas de calidad / puntuacion
        spark_sum("score").alias("score_total"),
        spark_round(avg("score"), 2).alias("score_promedio"),

        # Alcance e interaccion
        spark_sum("view_count").alias("total_vistas"),
        spark_sum("comment_count").alias("total_comentarios_recibidos"),

        # Efectividad: cuantas de sus preguntas tienen respuesta aceptada
        spark_sum(
            when(col("accepted_answer_id") > 0, 1).otherwise(0)
        ).alias("respuestas_aceptadas"),
    )
)

print(f"Usuarios con al menos un post: {df_posts_agg.count()}")

## 7. Cruzar con users_hist y aplicar reglas de negocio

Paso 2: LEFT JOIN sobre `users_hist`. Se conservan todos los usuarios
aunque no tengan posts en el periodo cargado, con sus contadores en 0.

Reglas de negocio aplicadas:
- **tasa_aceptacion_pct**: % de preguntas del usuario que obtuvieron respuesta aceptada
- **segmento_reputacion**: clasificacion segun la escala oficial de Stack Overflow
- **ratio_votos**: relacion upvotes/downvotes (mide la calidad percibida del usuario)

In [ ]:
print("Cruzando con users_hist y aplicando reglas de negocio...")

df_gold = (
    df_users
    .join(df_posts_agg, df_users["id"] == df_posts_agg["user_id"], "left")
    .drop(df_posts_agg["user_id"])   # eliminar columna duplicada del join

    # Rellenar nulos: usuarios sin posts quedan con metricas en 0
    .withColumn("total_posts",                 coalesce(col("total_posts"),                 lit(0)))
    .withColumn("total_preguntas",             coalesce(col("total_preguntas"),             lit(0)))
    .withColumn("total_respuestas",            coalesce(col("total_respuestas"),            lit(0)))
    .withColumn("score_total",                 coalesce(col("score_total"),                 lit(0)))
    .withColumn("score_promedio",              coalesce(col("score_promedio"),              lit(0.0)))
    .withColumn("total_vistas",                coalesce(col("total_vistas"),                lit(0)))
    .withColumn("total_comentarios_recibidos", coalesce(col("total_comentarios_recibidos"), lit(0)))
    .withColumn("respuestas_aceptadas",        coalesce(col("respuestas_aceptadas"),        lit(0)))

    # Regla 1: tasa de aceptacion (evitar division por cero)
    .withColumn(
        "tasa_aceptacion_pct",
        spark_round(
            when(
                col("total_preguntas") > 0,
                col("respuestas_aceptadas") / col("total_preguntas") * 100
            ).otherwise(lit(0.0)),
            2
        )
    )

    # Regla 2: segmento segun reputacion de Stack Overflow
    .withColumn(
        "segmento_reputacion",
        when(col("reputation") >= 10000, lit("Experto"))
        .when(col("reputation") >= 1000,  lit("Avanzado"))
        .when(col("reputation") >= 100,   lit("Intermedio"))
        .otherwise(lit("Principiante"))
    )

    # Regla 3: ratio votos positivos vs negativos (evitar division por cero)
    .withColumn(
        "ratio_votos",
        spark_round(
            when(
                col("down_votes") > 0,
                col("up_votes") / col("down_votes")
            ).otherwise(col("up_votes").cast("double")),
            2
        )
    )

    # Renombrar clave principal para consistencia Gold
    .withColumnRenamed("id", "user_id")

    # Columna de auditoria requerida por la capa Gold
    .withColumn("fecha_cargue", current_date())

    # Orden de columnas final
    .select(
        "user_id",
        "display_name",
        "reputation",
        "segmento_reputacion",
        "location",
        "creation_date",
        "last_access_date",
        "views",
        "up_votes",
        "down_votes",
        "ratio_votos",
        "total_posts",
        "total_preguntas",
        "total_respuestas",
        "respuestas_aceptadas",
        "tasa_aceptacion_pct",
        "score_total",
        "score_promedio",
        "total_vistas",
        "total_comentarios_recibidos",
        "fecha_cargue",
    )
)

print(f"Registros en tabla Gold: {df_gold.count()}")
df_gold.printSchema()

In [ ]:
df_gold.show(10, truncate=False)

## 8. Escribir en Gold con merge

In [ ]:
merge_into_iceberg(df_gold, GOLD_METRICS_TABLE, key_col="user_id")
print("Gold cant_post_x_user_hist completado.")

## 9. Verificacion y resumen final

In [ ]:
df_result = spark.read.format("iceberg").load(GOLD_METRICS_TABLE)

total = df_result.count()

print("=" * 55)
print("RESUMEN GOLD")
print("=" * 55)
print(f"  Tabla : {GOLD_METRICS_TABLE}")
print(f"  Filas : {total}")
print("=" * 55)

df_result.show(10, truncate=False)

# Distribucion por segmento (validacion rapida de reglas de negocio)
print("Distribucion por segmento_reputacion:")
df_result.groupBy("segmento_reputacion").count().orderBy("count").show()

print("Gold aggregation completado correctamente.")